In [1]:
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "distilbert/distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

print("Model:", model_id)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)

c:\Users\sergi\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 76/76 [00:00<00:00, 162.58it/s]

Model: distilbert/distilgpt2
PyTorch: 2.15.0+cpu
Transformers: 5.23.0


In [2]:
examples = [
    "Hello, world!",
    "The cat sat on the mat.",
    "Tokenization is unbelievable.",
]

for text in examples:
    ids = tokenizer(text)["input_ids"]
    tokens = tokenizer.convert_ids_to_tokens(ids)
    decoded = tokenizer.decode(ids)
    print("Text:", text)
    print("Tokens:", tokens)
    print("Token IDs:", ids)
    print("Decoded:", decoded)
    print()

Text: Hello, world!
Tokens: ['Hello', ',', 'Ġworld', '!']
Token IDs: [15496, 11, 995, 0]
Decoded: Hello, world!

Text: The cat sat on the mat.
Tokens: ['The', 'Ġcat', 'Ġsat', 'Ġon', 'Ġthe', 'Ġmat', '.']
Token IDs: [464, 3797, 3332, 319, 262, 2603, 13]
Decoded: The cat sat on the mat.

Text: Tokenization is unbelievable.
Tokens: ['Token', 'ization', 'Ġis', 'Ġunbelievable', '.']
Token IDs: [30642, 1634, 318, 24479, 13]
Decoded: Tokenization is unbelievable.



The word “Tokenization” was split into two tokens, one “Token” and the second “ization,” with the token IDs 30642 and 1634. Other words like “cat,” “world” and even the long word “unbelievable” each stayed as a single token. This happens because the tokenizer keeps common words whole but breaks less common words into smaller pieces that are in its vocabulary. This means the model does not always see a word as one unit, so one word can use up more than one token at a time.

In [5]:
def generate(model, tokenizer, prompt, max_new_tokens=50, **generate_kwargs):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             pad_token_id=tokenizer.eos_token_id,
                             **generate_kwargs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

prompt = "The best way to learn programming is"

for seed in [42, 42, 7]:
    torch.manual_seed(seed)
    text = generate(model, tokenizer, prompt, do_sample=True)
    print("Seed:", seed)
    print(text)
    print()

Seed: 42
The best way to learn programming is to follow the same rule as in your own language: to develop your own language and then create a new language for yourself. For this reason, I recommend doing what you'd like. Once you've been trained for this purpose, then simply experiment and

Seed: 42
The best way to learn programming is to follow the same rule as in your own language: to develop your own language and then create a new language for yourself. For this reason, I recommend doing what you'd like. Once you've been trained for this purpose, then simply experiment and

Seed: 7
The best way to learn programming is to sit down and learn, and practice with all of your friends, and to enjoy programming at home. This is also why you should keep up with great stuff.



I love you, and I hate you.
That is,



In [4]:
for temperature in [0.3, 1.0, 1.5]:
    torch.manual_seed(42)
    text = generate(model, tokenizer, prompt, do_sample=True, temperature=temperature)
    print("Temperature:", temperature)
    print(text)
    print()

Temperature: 0.3
The best way to learn programming is to learn from the people who have taught you.









































Temperature: 1.0
The best way to learn programming is to follow the same rule as in your own language: to develop your own language and then create a new language for yourself. For this reason, I recommend doing what you'd like. Once you've been trained for this purpose, then simply experiment and

Temperature: 1.5
The best way to learn programming is to follow some standards,” said John Danton, senior project and instructor and manager of the Web Learning & Learning Program at North Dakota State University. †



The state code states how programmers can create code without fear that programmers



I kept the prompt and seed the same. The only thing that changed was the temperature. At 0.3 the model wrote one short sentence and then only blank lines. At 1.0 it gave a longer answer that drifted off topic. At 1.5 it made up a quote from a named person as if it were real. A higher temperature makes the output come out more varied but also more likely to invent details.

In [6]:
prompts = [
    "The best way to learn programming is",
    "Here is my advice for a beginner. The best way to learn programming is",
    "What is the best way to learn programming? The answer is",
]

for p in prompts:
    torch.manual_seed(42)
    text = generate(model, tokenizer, p, do_sample=True)
    print("Prompt:", p)
    print(text)
    print()

Prompt: The best way to learn programming is
The best way to learn programming is to follow the same rule as in your own language: to develop your own language and then create a new language for yourself. For this reason, I recommend doing what you'd like. Once you've been trained for this purpose, then simply experiment and

Prompt: Here is my advice for a beginner. The best way to learn programming is
Here is my advice for a beginner. The best way to learn programming is to just ask a teacher.

Many beginner programmers, especially beginners like Erlang, have a deep understanding of programming. For this reason I‮m looking to provide you with instructions and tools to create a language as a language as quickly as

Prompt: What is the best way to learn programming? The answer is
What is the best way to learn programming? The answer is: there are a number of different ways that you can learn programming. Those may be the same kind of learning process that is the most important way for 

I asked for the same advice three different ways and kept everything else the same. The first prompt gave the same answer as before. When I added "for a beginner," the model said to ask a teacher, but after that then its writing stopped making sense. When I asked it as a question, the answer was vague and started talking about teaching instead of learning. Changing only the wording changed the answer so much.

One failure I saw was at temperature 1.5, where the model made up a quote from a named person at a university and presented it as if it were real. Nothing in the prompt mentioned this person, so the model invented it. In a customer-facing product, this could lead customers to believe false information or think a real person or school said something they never said. This can hurt the company's trust and possibly cause legal problems, so any product using this model would need people or checks to catch made-up facts.